# Exploring FEFLOW meshes with pyramids-mesh

**FEFLOW** (DHI) is a finite-element simulator for subsurface flow, heat and mass transport in groundwater. Its classic ASCII `.fem` file bundles the whole model, but the part `pyramids-mesh` cares about is the **mesh**: a `NODE` block listing element connectivity and a `COOR` block listing node coordinates. The `feflow` driver parses that pair and hands back a `pyramids.netcdf.ugrid.MeshDataset`, so a FEFLOW domain becomes an ordinary pyramids object.

The `.fem` format has drifted a lot across FEFLOW versions, and most real files in the wild are **not** the simple 2-D layout the driver supports: they are binary, or 3-D volume meshes, or use the newer variable-node tables. This notebook is therefore a two-sided lesson. First we read the one 2-D triangular file the driver **does** accept and tour it; then we walk the real corpus of files it **refuses** and read the explicit reason the driver gives for each -- so you can recognise an unsupported `.fem` before it surprises you.

## Setup

`MeshDataset` is the object every pyramids-mesh driver returns -- a wrapper around a 2D or 1D mesh. `UnsupportedMeshFormatError` is what a driver raises when it recognises a file but cannot read that particular variant; we import it so the negative section can catch it precisely. Importing from `pyramids` also places its vendored GDAL/PROJ data on the path for us, so there is no import-order ritual in a notebook. `add_tiles` would drape a basemap *when* a mesh carries a CRS (this one does not, as we will see).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from cleopatra.basemap.tiles import add_tiles

from pyramids.netcdf.ugrid import MeshDataset, UnsupportedMeshFormatError

## Locate the sample files

This notebook lives in `docs/examples/real-data/`, so the repository's `examples/data/real-data/` folder is **three** levels up. Paths inside a notebook are always written relative to the notebook itself, and pyramids' `read_file` takes a `pathlib.Path` directly -- we pass the `Path`, never `str(path)`. Every `.fem` file for this driver sits in the `feflow/` subfolder.

In [ ]:
DATA = Path("../../../examples/data/real-data")
FEFLOW = DATA / "feflow"
FEFLOW, FEFLOW.is_dir()

## What is in the folder

The folder ships a `MANIFEST.csv` documenting every file: its FEFLOW version, whether it is ASCII or binary, which connectivity section it uses (`NODE` vs the newer `VARNODE`), which coordinate section (`COOR` vs `XYZCOOR`), the element type, and -- crucially -- a `driver_reads` flag with a `driver_fail_reason`. We read it with pandas to see the shape of the corpus before touching any mesh.

In [ ]:
manifest = pd.read_csv(FEFLOW / "MANIFEST.csv")
manifest[["new_name", "encoding", "conn_section", "coord_section", "element", "driver_reads", "driver_fail_reason"]]

The `driver_reads` column tells the whole story: **exactly one** file is `True`. It is the lone `ascii` / `NODE` / `COOR` / `tri` row -- the classic 2-D triangular layout. Every other row is a distinct reason the driver stops: `binary` encoding, `COOR` carrying 3-D volume coordinates, or a `VARNODE`/`XYZCOOR` mixed-element table. We read the one supported file first.

In [ ]:
manifest["driver_reads"].value_counts().rename("n_files")

## Read the supported mesh

`MeshDataset.read_file(path, driver="feflow")` dispatches the file to the `feflow` driver, which parses the `NODE`/`COOR` pair and returns a `MeshDataset` -- the same type a native UGRID NetCDF file would produce. The supported sample is a small triangular mesh from an Intrepid Geophysics meshing tutorial.

In [ ]:
mesh_file = FEFLOW / "feflow_node-coor_tri_v6.2_15n_intrepid.fem"
ds = MeshDataset.read_file(mesh_file, driver="feflow")
type(ds).__name__

Reading a mesh is when cleopatra switches matplotlib to a headless backend. Right after the first read we let cleopatra pick the right backend for this environment -- the inline backend in a notebook, `Agg` in a plain script -- so every plot below embeds in the cell.

In [ ]:
from cleopatra.config import Config

Config.set_matplotlib_backend()

## Explore the dataset

A `MeshDataset` exposes summary properties for the whole mesh: `n_node` / `n_face` (size), `crs` / `epsg` (georeferencing, if the format recorded any), `bounds` (spatial extent as `xmin, ymin, xmax, ymax`), and `data_variable_names` (any attached fields).

In [ ]:
pd.Series({
    "n_node": ds.n_node,
    "n_face": ds.n_face,
    "crs": ds.crs,
    "epsg": ds.epsg,
    "bounds": tuple(round(float(b), 3) for b in ds.bounds),
    "data_variable_names": ds.data_variable_names,
})

**15 nodes, 18 faces** -- a tiny mesh, exactly what a tutorial domain should be. `crs` and `epsg` are `None`: the classic FEFLOW ASCII format stores raw model coordinates with no projection tag, so the `bounds` are plain model units (here a few-kilometre-wide block), not degrees. `data_variable_names` is empty -- the `.fem` mesh is geometry only; simulated heads and temperatures live in separate result files.

### Every element is a triangle

`ds.mesh.face_node_connectivity.data` is the `(n_face, k)` table of node indices per face, where `k` is the widest polygon. A cell with fewer nodes than `k` is padded with `-1` (fill). Counting the non-negative entries per row gives each face's degree -- its number of nodes.

In [ ]:
conn = ds.mesh.face_node_connectivity.data
degree = (conn >= 0).sum(axis=1)
vals, counts = np.unique(degree, return_counts=True)
pd.Series({
    "connectivity_shape": conn.shape,
    "degree_distribution": dict(zip(vals.tolist(), counts.tolist())),
    "fill_slots(-1)": int((conn == -1).sum()),
})

All 18 elements are **3-node triangles** with no `-1` fill -- a clean, uniform triangulation. Triangles are the FEFLOW 2-D default, and being purely triangular matters later: `circumcenters()` is defined only for triangular meshes.

### The underlying geometry

`ds.mesh` is the `Mesh2d` holding the raw arrays: `node_x` / `node_y` (coordinates), the connectivity table, `face_areas` (one area per cell) and `face_centroids` (shape `(2, n_face)`). Every geometric quantity is derived from the node coordinates plus the connectivity.

In [ ]:
mesh = ds.mesh
areas = mesh.face_areas
pd.Series({
    "n_node": int(mesh.node_x.shape[0]),
    "x_range": (round(float(mesh.node_x.min()), 1), round(float(mesh.node_x.max()), 1)),
    "y_range": (round(float(mesh.node_y.min()), 1), round(float(mesh.node_y.max()), 1)),
    "area_min": round(float(areas.min()), 1),
    "area_max": round(float(areas.max()), 1),
})

Coordinates run in the thousands on both axes and cell areas are in the millions -- consistent with raw metric model units (roughly a 7-25 km block), not degrees. There is no CRS to tell us which metric system, which is exactly why the basemap guard `if ds.epsg:` never fires below.

## The wireframe

The simplest view needs no data: just draw the cell edges. `plot_outline()` renders the mesh skeleton through cleopatra and is the fastest check that a file read in correctly. We keep the `add_tiles` guard so a CRS-bearing mesh would get a basemap -- here it is skipped because `epsg` is `None`.

In [ ]:
g = ds.plot_outline()
if ds.epsg:
    add_tiles(g.ax, crs=ds.epsg)

A compact fan of 18 triangles over 15 nodes -- the whole tutorial domain at a glance. Because the mesh is so small you can see individual elements, which makes it a good teaching case for the connectivity and GIS operations that follow.

## A data map: per-cell area

To draw a *filled* map we need a value per cell. This mesh carries no fields, so we **synthesise** one -- each cell's area -- and attach it with `with_variable(..., location="face")`, which returns a **new** dataset and leaves `ds` untouched. Any real per-cell field (head, temperature, flow depth) would plot through exactly the same call.

In [ ]:
d2 = ds.with_variable("cell_area", ds.mesh.face_areas, location="face")
d2.data_variable_names

In [ ]:
g = d2.plot("cell_area", cmap="viridis", colorbar=True, title="Per-cell area")
if ds.epsg:
    add_tiles(g.ax, crs=ds.epsg)

Each triangle is now shaded by its area: the larger outer elements stand out against the finer ones. This is the generic pyramids data-map path -- nothing about it is FEFLOW-specific. `summary()` reports the statistics of every attached field.

In [ ]:
d2.summary()

A single `cell_area` row with `count = 18` -- one value per triangle -- summarising the spread between the smallest and largest elements. On the bare `ds` (no fields) this frame would be empty.

## GIS operations, for free

Because the mesh is a first-class pyramids object, the pyramids GIS layer applies directly -- none of these methods live in `pyramids-mesh`, which only parses the file. We tour three.

### Faces as a GeoDataFrame

`to_geodataframe()` turns every face into a polygon row of a geopandas `GeoDataFrame` -- the standard vector structure for export to shapefile/GeoPackage or for spatial joins. With 18 triangles the whole table fits on screen.

In [ ]:
gdf = ds.to_geodataframe()
gdf.head()

One triangular cell per row. The CRS column is unset (matching `epsg = None`); you would assign the known model CRS before exporting for use in other GIS tools.

### Bounding polygon

`bounding_polygon()` dissolves the cells into a single shapely `Polygon` of the meshed footprint -- the gateway to the shapely/geopandas ecosystem for clipping and spatial joins.

In [ ]:
poly = ds.bounding_polygon()
pd.Series({"type": poly.geom_type, "bounds": tuple(round(float(b), 1) for b in poly.bounds)})

A plain shapely `Polygon` spanning the model extent -- ready for clipping or export. Its bounds match the mesh `bounds` reported earlier.

### Circumcenters

`circumcenters()` returns an `(n_face, 2)` array of per-cell circumcenters -- the points a finite-volume scheme places unknowns at. It is defined **only for triangular meshes**, which is precisely why it works here and would not on the 3-D volume files later in the corpus.

In [ ]:
cc = ds.circumcenters()
cc.shape, np.round(cc[:3], 1)

One circumcenter per triangle (`(18, 2)`). That this call succeeds at all is a property of the mesh being purely triangular -- the single success criterion the supported `.fem` layout happens to guarantee.

## The negative corpus: what the driver refuses, and why

Thirteen of the fourteen files cannot be read. That is not a bug -- it is the driver being honest about the one layout it supports and declining the rest with an explicit message. The refusals fall into three classes:

| Class | Representative file | MANIFEST reason | Why the driver stops |
|---|---|---|---|
| **Binary `.fem`** | `feflow_binary_prism_v6.2_1.1kn_subsidence-deleeuw.fem` | `no NODE section` | The mesh lives in a binary `BINARY` block, not the ASCII `NODE`/`COOR` sections this reader parses. |
| **3-D volume (NODE + COOR)** | `feflow_node-coor_hex_v6.2_144n_ogs-cube.fem` | `COOR carries` | The `COOR` block carries 3-D volume coordinates (hex/prism), not the flat 2-D node list the reader expects. |
| **VARNODE / XYZCOOR mixed** | `feflow_varnode-xyzcoor_mixed_v7.0_13n_ogs-tet.fem` | `nodes_per_element` | The newer variable-node table encodes mixed element types the reader does not unpack. |

Each file raises `UnsupportedMeshFormatError`. We read one representative of each class, catch the error, and keep the message the driver gives.

In [ ]:
representatives = {
    "binary": "feflow_binary_prism_v6.2_1.1kn_subsidence-deleeuw.fem",
    "3-D NODE+COOR volume": "feflow_node-coor_hex_v6.2_144n_ogs-cube.fem",
    "VARNODE/XYZCOOR mixed": "feflow_varnode-xyzcoor_mixed_v7.0_13n_ogs-tet.fem",
}

rows = []
for cls, name in representatives.items():
    try:
        MeshDataset.read_file(FEFLOW / name, driver="feflow")
        reason = "(unexpectedly read without error)"
        error = ""
    except UnsupportedMeshFormatError as exc:
        error = type(exc).__name__
        # the driver echoes the path twice; keep the explanatory tail after the last filename
        reason = str(exc).rsplit(name + "'", 1)[-1].strip().rstrip('")').strip()
    rows.append({"class": cls, "file": name, "error": error, "reason": reason})

pd.set_option("display.max_colwidth", None)
pd.DataFrame(rows).set_index("class")

Every file raised `UnsupportedMeshFormatError`, and each message pins down the exact reason -- matching the `driver_fail_reason` in the MANIFEST:

- **Binary** (`no NODE section`): the file declares a `BINARY` section, so the mesh is stored in a binary block rather than the ASCII `NODE`/`COOR` sections -- only the classic ASCII `.fem` is supported.
- **3-D NODE+COOR volume** (`COOR carries`): the `COOR` block carries a node count that is not a flat 2-D list -- it implies a 3-D hex/prism volume, which the 2-D reader will not flatten.
- **VARNODE/XYZCOOR mixed** (`nodes_per_element`): the `DIMENS` header advertises a variable-node table with mixed element types that the reader does not unpack.

A caught `UnsupportedMeshFormatError` with one of these messages is the signal that a `.fem` is a real FEFLOW file in a variant the driver does not yet read -- not that the file is corrupt.

## Takeaway

- The `feflow` driver reads the **classic ASCII `.fem` layout** -- a `NODE` connectivity block plus a `COOR` node-coordinate block describing a flat **2-D triangular** mesh -- into a pyramids `MeshDataset`.
- The one supported sample is a small all-triangular mesh (15 nodes, 18 faces, degree 3, no `-1` fill) with **no CRS** (raw model units) and no attached fields. Being triangular is what lets `circumcenters()` work.
- Once read, the full pyramids GIS layer applies unchanged -- `to_geodataframe`, `bounding_polygon`, `circumcenters`, a synthesised data map -- while CRS-dependent steps are skipped because `epsg` is `None`.
- The other thirteen real files are **current gaps**, and the driver says so explicitly with `UnsupportedMeshFormatError`: **binary** `.fem`, **3-D volume** `NODE`/`COOR` meshes (hex/prism), and **`VARNODE`/`XYZCOOR`** mixed-element tables. Recognising those three messages tells you an unsupported-but-valid FEFLOW file from a broken one.